The goal of this example is to show how to run a case with changing winddirections

In [ ]:
# Import some packages:
import numpy as np
import matplotlib.pyplot as plt
from WindGym import WindFarmEnv
from py_wake.examples.data.hornsrev1 import V80 as WindTurbine
from WindGym.Agents import GreedyAgent, PyWakeAgent
import copy

def make_config():
    # Base configuration dictionary
    config_dict = {
        "yaw_init": "Random",
        "BaseController": "Local",
        "ActionMethod": "wind",
        "Track_power": False,
        "farm": {
            "yaw_min": -40,
            "yaw_max": 40,
        },
        "wind": {
            "ws_min": 10,
            "ws_max": 10,
            "TI_min": 0.06,
            "TI_max": 0.06,
            "wd_min": 270,
            "wd_max": 270,
        },
        "act_pen": {"action_penalty": 0.0, "action_penalty_type": "Change"},
        "power_def": {"Power_reward": "Power_avg", "Power_avg": 1, "Power_scaling": 1.0},
        "mes_level": {
            "turb_ws": True,
            "turb_wd": True,
            "turb_TI": False,
            "turb_power": False,
            "farm_ws": False,
            "farm_wd": False,
            "farm_TI": False,
            "farm_power": False,
        },
        "ws_mes": {
            "ws_current": True,
            "ws_rolling_mean": False,
            "ws_history_N": 1,
            "ws_history_length": 1,
            "ws_window_length": 1,
        },
        "wd_mes": {
            "wd_current": True,
            "wd_rolling_mean": False,
            "wd_history_N": 1,
            "wd_history_length": 1,
            "wd_window_length": 1,
        },
        "yaw_mes": {
            "yaw_current": True,
            "yaw_rolling_mean": False,
            "yaw_history_N": 1,
            "yaw_history_length": 1,
            "yaw_window_length": 1,
        },
        "power_mes": {
            "power_current": False,
            "power_rolling_mean": False,
            "power_history_N": 1,
            "power_history_length": 1,
            "power_window_length": 1,
        },
    }

    return config_dict

from WindGym.utils.generate_layouts import generate_square_grid

ModuleNotFoundError: No module named 'windgym'

We need to define a wind_direction function. Down below are 3 examples.

Note that the function could also be 'random' in that you could make one the samples some rng, and then chooses based on that

In [ ]:
def make_winddirection(t):
    # Wind direction function for numpy array input
    t = np.asarray(t)  # Ensure input is a numpy array
    return np.where(t < 100, 270, 270 + 10 * np.sin(2 * np.pi * (t - 100) / 1000))


In [ ]:
def make_winddirection_alternative(t):
    # Wind direction function for numpy array input
    t = np.asarray(t)  # Ensure input is a numpy array
    return np.where(t <= 100, 270, np.minimum(270 + (t - 100)*0.14, 300))

In [ ]:
def make_winddirection_360(t):
    # Wind direction function for numpy array input
    t = np.asarray(t)  # Ensure input is a numpy array
    return (270 + 270 * t/2000 ) % 360

In [ ]:
# Here is a plot of the wind direction over time using the third function:
time_plot = np.arange(0, 1000)
wd_plot = make_winddirection_360(time_plot)

plt.plot(time_plot, wd_plot)
plt.xlabel("Time (s)")
plt.ylabel("Wind Direction (degrees)")
plt.title("Wind Direction Over Time")
plt.grid()
plt.show()


Now we define the environment as usual

In [ ]:
# Turbine posistions
x_pos, y_pos = generate_square_grid(turbine=WindTurbine(), nx=3, ny=1, xDist=5, yDist=5)

# The environment
env = WindFarmEnv(
    turbine = WindTurbine(),
    yaw_init='Zeros',
    x_pos=x_pos,
    y_pos=y_pos,
    n_passthrough=100,
    config=make_config(),
    dt_sim=5,
    dt_env=10,
    yaw_step_sim=5,
    wd_function=make_winddirection_360,  # Use the desired wind direction function here
    turbtype="Random",
    max_turb_move=500,                  # This is the maximum distance a turbine can move pr step in meters. I just set something unrealistic high here
    )


# We need an agent to interact with the environment.
# here are 2 versions.
agent = GreedyAgent(env=env, type="local", 
                    yaw_min=env.yaw_min, yaw_max=env.yaw_max,
                    yaw_step=env.yaw_step_sim)

pywake_agent = PyWakeAgent(x_pos=x_pos, y_pos=y_pos, yaw_max=env.yaw_max, yaw_min=env.yaw_min, 
                           turbine=WindTurbine(),
                           env=env,
                           look_up=True,
                            )

In [ ]:
# Plot the farm initially:
env.plot_farm(fix_turbines=True)

In [ ]:
FIX_TURBINES=True
SAVE_FIGS=False
file_name = "flow"

wd_plot_1 = []
wd_plot_2 = []
wd_plot_3 = []
time_plot = []
yaw_angles = []
power_plot = []


obs, info = env.reset(seed=20)

pywake_agent.update_wind(wind_speed=env.ws,
                         wind_direction=env.wd,
                         TI=env.ti)

wd_plot_1.append(copy.copy(env.fs._wind_direction))
wd_plot_2.append(copy.copy(env.fs.get_wind_direction(xyz=(-200,0,0), include_wakes=False)))
wd_plot_3.append(copy.copy(env.wd))
time_plot.append(copy.copy(env.fs.time))
yaw_angles.append(copy.copy(env.fs.windTurbines.yaw))
power_plot.append(copy.copy(env.fs.windTurbines.power()))

if SAVE_FIGS:
    # If we want to the figure
    env.plot_farm(fix_turbines=FIX_TURBINES)

    ax = plt.gca()
    fig = plt.gcf()

    fig.savefig(file_name+"_{:05d}.png".format(int(env.fs.time)))
    plt.close(fig)

for _ in range(250):
    # Actions from the greedy agent. Overwritten by pywake agent below.
    action, _ = agent.predict(obs, info)
    # action = np.zeros_like(env.action_space.sample())

    # PyWake agent needs the 'correct' wind conditions
    pywake_agent.update_wind(wind_speed=env.ws,
                         wind_direction=env.wd,
                         TI=env.ti)
    # The pywake agent action
    action, _ = pywake_agent.predict(obs, info)

    obs, reward, terminated, truncated, info = env.step(action)

    # Add data to plots
    wd_plot_1.append(copy.copy(env.fs._wind_direction))
    wd_plot_2.append(copy.copy(env.fs.get_wind_direction(xyz=(-200,0,0), include_wakes=False)))
    wd_plot_3.append(copy.copy(env.wd))
    time_plot.append(copy.copy(env.fs.time))
    yaw_angles.append(copy.copy(env.fs.windTurbines.yaw))
    power_plot.append(copy.copy(env.fs.windTurbines.power()))

    if SAVE_FIGS:
        env.plot_farm(fix_turbines=FIX_TURBINES)
        ax = plt.gca()
        fig = plt.gcf()

        # # Save the figure:9
        fig.savefig(file_name+"_{:05d}.png".format(int(env.fs.time)))

        # close the plot
        plt.close(fig)

wd_plot_1 = np.array(wd_plot_1)
wd_plot_2 = np.array(wd_plot_2)
wd_plot_3 = np.array(wd_plot_3)
time_plot = np.array(time_plot)
yaw_angles = np.array(yaw_angles)
power_plot = np.array(power_plot)



In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

ax[0].plot(time_plot, wd_plot_1, label='fs._wind_direction')
ax[0].plot(time_plot, wd_plot_2, label='fs.get_wind_direction at (-200,0,0)')
ax[0].plot(time_plot, wd_plot_3, label='env.wd')
ax[0].set_xlabel("Time (s)")
ax[0].set_ylabel("Wind Direction (degrees)")
ax[0].set_title("Wind Direction from Different Sources Over Time")
ax[0].legend()
ax[0].grid()


ax[1].plot(time_plot, yaw_angles)
ax[1].set_xlabel("Time (s)")
ax[1].set_ylabel("Yaw Angles (degrees)")
ax[1].set_title("Yaw Angles of Turbines Over Time")
ax[1].grid()
ax[1].legend([f'Turbine {i+1}' for i in range(yaw_angles.shape[1])])


ax[2].plot(time_plot, wd_plot_3[:, np.newaxis]-yaw_angles)
ax[2].set_xlabel("Time (s)")
ax[2].set_ylabel("Yaw angles global \n refference frame (degrees)")
ax[2].set_title("Yaw Angles Relative to Wind Direction Over Time")
ax[2].grid()
ax[2].legend([f'Turbine {i+1}' for i in range(yaw_angles.shape[1])])


ax[3].plot(time_plot, power_plot)
ax[3].set_xlabel("Time (s)")
ax[3].set_ylabel("Power Output (W)")
ax[3].set_title("Power Output of Turbines Over Time")
ax[3].grid()
# legend:
ax[3].legend([f'Turbine {i+1}' for i in range(power_plot.shape[1])])


plt.tight_layout()
plt.show()